# Fase 1B — Preprocesamiento para Transformers (IroSvA)

Este notebook es **independiente** del preprocessing de Machine Learning.

Modelos objetivo:

- **BETO cased** — `dccuchile/bert-base-spanish-wwm-cased`
- **DistilBETO uncased** — `dccuchile/distilbert-base-spanish-uncased`
- **RoBERTuito uncased** — `pysentimiento/robertuito-base-uncased`

También se deja identificado `pysentimiento/robertuito-base-deacc` como checkpoint alternativo para una comparación posterior.

## Principios

- Se parte siempre de `MESSAGE` original.
- No se utiliza `MESSAGE_CLEAN`, stemming ni lematización del notebook de ML.
- Se mantienen stopwords, puntuación y estructura de la oración.
- El test oficial se procesa para disponer del archivo final, pero **no se utiliza para seleccionar preprocessing, max_length ni checkpoint**.
- BETO y DistilBETO compararán posteriormente dos alternativas de texto:
  - `BASE`: normalización mínima.
  - `SOCIAL`: normalización específica de redes sociales.
- RoBERTuito usa su `preprocess_tweet` oficial.


## 1. Librerías y configuración

In [2]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import emoji

try:
    from transformers import AutoTokenizer
    TRANSFORMERS_DISPONIBLE = True
except ImportError:
    AutoTokenizer = None
    TRANSFORMERS_DISPONIBLE = False

try:
    from pysentimiento.preprocessing import preprocess_tweet
    PYSENTIMIENTO_DISPONIBLE = True
except ImportError:
    preprocess_tweet = None
    PYSENTIMIENTO_DISPONIBLE = False

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "transformers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
}

ROBERTUITO_ALTERNATIVO = "pysentimiento/robertuito-base-deacc"

print("transformers disponible:", TRANSFORMERS_DISPONIBLE)
print("pysentimiento disponible:", PYSENTIMIENTO_DISPONIBLE)
print()
for modelo, checkpoint in CHECKPOINTS.items():
    print(f"{modelo:12} -> {checkpoint}")
print(f"{'RoBERTuito alt.':12} -> {ROBERTUITO_ALTERNATIVO}")


transformers disponible: False
pysentimiento disponible: False

BETO         -> dccuchile/bert-base-spanish-wwm-cased
DistilBETO   -> dccuchile/distilbert-base-spanish-uncased
RoBERTuito   -> pysentimiento/robertuito-base-uncased
RoBERTuito alt. -> pysentimiento/robertuito-base-deacc


## 2. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar duplicados se verifica:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Solo los duplicados exactos de texto se deduplican después de la auditoría, manteniendo una única copia.  
Esto replica el criterio utilizado en el preprocessing de ML para mantener ambos experimentos uniformizados.


In [3]:
def auditar_duplicados_train(df, variante):
    grupos = (
        df.groupby("MESSAGE", dropna=False)["IS_IRONIC"]
        .agg(["size", "nunique"])
        .reset_index()
    )

    repetidos = grupos[
        grupos["size"] > 1
    ].copy()

    conflictos = repetidos[
        repetidos["nunique"] > 1
    ].copy()

    mismos_labels = repetidos[
        repetidos["nunique"] == 1
    ].copy()

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismos_labels),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(
            conflictos["MESSAGE"].tolist()
        ),
        "conflictos_resumen": conflictos,
    }


def cargar_train():
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in ["mx", "es", "cu"]:
        df = pd.read_csv(
            DATA_DIR / f"irosva.{variante}.training.csv"
        )
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(
            df,
            variante,
        )

        mensajes_conflicto = info[
            "mensajes_conflicto"
        ]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[
            df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        if not df_conflictos.empty:
            df_conflictos[
                "MOTIVO_EXCLUSION"
            ] = "mismo_texto_etiquetas_contradictorias"

            conflictos_detalle.append(
                df_conflictos
            )

            print(
                f"\n{variante.upper()} — conflictos de etiqueta:"
            )

            display(
                df_conflictos[
                    [
                        "ID",
                        "MESSAGE",
                        "IS_IRONIC",
                        "VARIANTE",
                    ]
                ].sort_values("MESSAGE")
            )

        # 1. Eliminar completamente textos con etiquetas contradictorias.
        df_sin_conflictos = df[
            ~df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        # 2. En duplicados restantes, las etiquetas son iguales:
        #    conservar una única copia.
        antes_dedup = len(df_sin_conflictos)

        df_limpio = (
            df_sin_conflictos
            .drop_duplicates(
                subset="MESSAGE",
                keep="first",
            )
            .reset_index(drop=True)
        )

        filas_dedup_mismo_label = (
            antes_dedup
            - len(df_limpio)
        )

        auditoria.append({
            "variante": variante,
            "n_original": info["n_original"],
            "textos_repetidos": info["n_textos_repetidos"],
            "conflictos_texto": info["n_conflictos_etiqueta"],
            "filas_retiradas_conflicto": len(df_conflictos),
            "filas_duplicadas_misma_etiqueta_retiradas": filas_dedup_mismo_label,
            "n_tras_control_duplicados": len(df_limpio),
        })

        print(
            f"{variante}: "
            f"{info['n_original']} originales | "
            f"{len(df_conflictos)} filas retiradas por conflicto | "
            f"{filas_dedup_mismo_label} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )

        frames.append(
            df_limpio
        )

    df_train = pd.concat(
        frames,
        ignore_index=True,
    )

    df_auditoria = pd.DataFrame(
        auditoria
    )

    if conflictos_detalle:
        df_conflictos_train = pd.concat(
            conflictos_detalle,
            ignore_index=True,
        )
    else:
        df_conflictos_train = pd.DataFrame()

    return (
        df_train,
        df_auditoria,
        df_conflictos_train,
    )


(
    df_train,
    df_auditoria_train,
    df_conflictos_train,
) = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(
    "\nTrain tras control de duplicados/conflictos:",
    len(df_train),
)

print(
    df_train["VARIANTE"]
    .value_counts()
    .sort_index()
)

print("\nDistribución de clases:")
print(
    df_train
    .groupby("VARIANTE")["IS_IRONIC"]
    .value_counts()
)


mx: 2400 originales | 0 filas retiradas por conflicto | 1 duplicados de misma etiqueta retirados | 2399 restantes

ES — conflictos de etiqueta:


,ID,MESSAGE,IS_IRONIC,VARIANTE
762,3bb2e28f5c0f1a0dd7fbcbed779ff957,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,1,es
916,5353523b4ac97aff03c46cb3d11d4d6a,Están pasando mas cosas ?\r\n\r\nUn hijo de ju...,0,es


es: 2400 originales | 2 filas retiradas por conflicto | 1 duplicados de misma etiqueta retirados | 2397 restantes
cu: 2400 originales | 0 filas retiradas por conflicto | 0 duplicados de misma etiqueta retirados | 2400 restantes

AUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS


,variante,n_original,textos_repetidos,conflictos_texto,filas_retiradas_conflicto,filas_duplicadas_misma_etiqueta_retiradas,n_tras_control_duplicados
0,mx,2400,1,0,0,1,2399
1,es,2400,2,1,2,1,2397
2,cu,2400,0,0,0,0,2400



Train tras control de duplicados/conflictos: 7196
VARIANTE
cu    2400
es    2397
mx    2399
Name: count, dtype: int64

Distribución de clases:
VARIANTE  IS_IRONIC
cu        0            1600
          1             800
es        0            1598
          1             799
mx        0            1599
          1             800
Name: count, dtype: int64


## 3. Carga del test oficial

El test se carga y procesa únicamente para dejar preparada la entrada final.  
**No debe utilizarse en análisis de tokenización, selección BASE/SOCIAL, elección de max_length ni elección de checkpoint.**


In [4]:
def cargar_test(variante):
    df_texto = pd.read_csv(
        DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv"
    )
    df_truth = pd.read_csv(
        DATA_DIR / f"irosva.{variante}.test.truth.csv"
    )

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(
        df_truth[["ID", "IS_IRONIC"]],
        on="ID",
        how="inner",
    )

    df["VARIANTE"] = variante
    return df


df_test = pd.concat(
    [cargar_test(v) for v in ["mx", "es", "cu"]],
    ignore_index=True,
)

print("Test total:", len(df_test))
print(df_test["VARIANTE"].value_counts().sort_index())


Test total: 1800
VARIANTE
cu    600
es    600
mx    600
Name: count, dtype: int64


## 4. Auditoría e independencia Train ↔ Test

El **test oficial se mantiene intacto**.

Se comprueba:

- si existen duplicados internos en test;
- si algún `MESSAGE` del train aparece exactamente en el test;
- si las etiquetas train/test coinciden para esos textos.

Para evitar que un Transformer pueda memorizar un texto que luego aparece en evaluación,
las coincidencias exactas se **retiran únicamente del train** y se guardan en un archivo
de auditoría. El test oficial no se modifica.

Esta decisión debe documentarse en la metodología porque el train final difiere
mínimamente del split original de IroSvA.


In [5]:
def auditar_train_test(df_train, df_test):
    filas = []
    coincidencias_detalle = {}

    for variante in ["mx", "es", "cu"]:
        tr = df_train[
            df_train["VARIANTE"] == variante
        ].copy()

        te = df_test[
            df_test["VARIANTE"] == variante
        ].copy()

        dups_test = int(
            te.duplicated(
                subset="MESSAGE"
            ).sum()
        )

        mensajes_train = set(
            tr["MESSAGE"]
            .fillna("")
            .astype(str)
        )

        mensajes_test = set(
            te["MESSAGE"]
            .fillna("")
            .astype(str)
        )

        comunes = (
            mensajes_train
            & mensajes_test
        )

        # Mostrar etiquetas de ambos splits para cada coincidencia.
        detalle_train = tr[
            tr["MESSAGE"]
            .fillna("")
            .astype(str)
            .isin(comunes)
        ][
            ["ID", "MESSAGE", "IS_IRONIC"]
        ].rename(
            columns={
                "ID": "ID_TRAIN",
                "IS_IRONIC": "LABEL_TRAIN",
            }
        )

        detalle_test = te[
            te["MESSAGE"]
            .fillna("")
            .astype(str)
            .isin(comunes)
        ][
            ["ID", "MESSAGE", "IS_IRONIC"]
        ].rename(
            columns={
                "ID": "ID_TEST",
                "IS_IRONIC": "LABEL_TEST",
            }
        )

        detalle = detalle_train.merge(
            detalle_test,
            on="MESSAGE",
            how="inner",
        )

        if not detalle.empty:
            detalle["MISMA_ETIQUETA"] = (
                detalle["LABEL_TRAIN"]
                == detalle["LABEL_TEST"]
            )

        coincidencias_detalle[
            variante
        ] = detalle

        filas.append({
            "variante": variante,
            "train_n_antes": len(tr),
            "test_n": len(te),
            "duplicados_test": dups_test,
            "coincidencias_train_test": len(comunes),
        })

    return (
        pd.DataFrame(filas),
        coincidencias_detalle,
    )


df_auditoria_train_test, coincidencias_train_test = (
    auditar_train_test(
        df_train,
        df_test,
    )
)

print("AUDITORÍA TRAIN ↔ TEST")
display(df_auditoria_train_test)

for variante in ["mx", "es", "cu"]:
    detalle = coincidencias_train_test[variante]

    if len(detalle) > 0:
        print(
            f"\nCoincidencias exactas en {variante.upper()}:"
        )
        display(detalle)


# ============================================================
# RETIRAR COINCIDENCIAS DEL TRAIN — TEST OFICIAL INTACTO
# ============================================================

mensajes_test_por_variante = {
    variante: set(
        df_test.loc[
            df_test["VARIANTE"] == variante,
            "MESSAGE",
        ]
        .fillna("")
        .astype(str)
    )
    for variante in ["mx", "es", "cu"]
}

filas_excluidas = []
frames_train_filtrado = []

for variante in ["mx", "es", "cu"]:
    subset = df_train[
        df_train["VARIANTE"] == variante
    ].copy()

    mensajes_test = mensajes_test_por_variante[
        variante
    ]

    mask_solapamiento = (
        subset["MESSAGE"]
        .fillna("")
        .astype(str)
        .isin(mensajes_test)
    )

    excluidos = subset[
        mask_solapamiento
    ].copy()

    if not excluidos.empty:
        excluidos["MOTIVO_EXCLUSION"] = (
            "coincidencia_exacta_train_test"
        )
        filas_excluidas.append(
            excluidos
        )

    subset = subset[
        ~mask_solapamiento
    ].reset_index(drop=True)

    frames_train_filtrado.append(
        subset
    )

df_train = pd.concat(
    frames_train_filtrado,
    ignore_index=True,
)

if filas_excluidas:
    df_train_excluidos = pd.concat(
        filas_excluidas,
        ignore_index=True,
    )
else:
    df_train_excluidos = pd.DataFrame()

print(
    "\nTRAIN FINAL tras retirar coincidencias con test:"
)
print(
    df_train["VARIANTE"]
    .value_counts()
    .sort_index()
)

print(
    "\nRegistros retirados del train:",
    len(df_train_excluidos),
)

if len(df_train_excluidos) > 0:
    display(
        df_train_excluidos[
            [
                "ID",
                "MESSAGE",
                "IS_IRONIC",
                "VARIANTE",
                "MOTIVO_EXCLUSION",
            ]
        ]
    )


AUDITORÍA TRAIN ↔ TEST


,variante,train_n_antes,test_n,duplicados_test,coincidencias_train_test
0,mx,2399,600,0,2
1,es,2397,600,0,1
2,cu,2400,600,0,0



Coincidencias exactas en MX:


,ID_TRAIN,MESSAGE,LABEL_TRAIN,ID_TEST,LABEL_TEST,MISMA_ETIQUETA
0,910b1a08a43e051796edf541480ea2ae,No lo puede evitar teacher. Nacio perversa de...,0,910b1a08a43e051796edf541480ea2ae,0,True
1,61a69eb1aa7c10192d9b2e7c06927ab2,Qué falta de seriedad,0,61a69eb1aa7c10192d9b2e7c06927ab2,0,True



Coincidencias exactas en ES:


,ID_TRAIN,MESSAGE,LABEL_TRAIN,ID_TEST,LABEL_TEST,MISMA_ETIQUETA
0,515135acbbd86f0d07f653a471ad2b23,#YoConAlbert Einstein,1,807327f4adead1d48fc2317a1f70588b,1,True



TRAIN FINAL tras retirar coincidencias con test:
VARIANTE
cu    2400
es    2396
mx    2397
Name: count, dtype: int64

Registros retirados del train: 3


,ID,MESSAGE,IS_IRONIC,VARIANTE,MOTIVO_EXCLUSION
0,910b1a08a43e051796edf541480ea2ae,No lo puede evitar teacher. Nacio perversa de...,0,mx,coincidencia_exacta_train_test
1,61a69eb1aa7c10192d9b2e7c06927ab2,Qué falta de seriedad,0,mx,coincidencia_exacta_train_test
2,515135acbbd86f0d07f653a471ad2b23,#YoConAlbert Einstein,1,es,coincidencia_exacta_train_test


## 5. Features lingüísticas opcionales

Se conservan las mismas 9 features del experimento ML, calculadas **desde el texto original**.

No se incorporarán en el primer baseline Transformer. Se guardan para un experimento posterior:

**Transformer puro vs Transformer + features lingüísticas**.


In [6]:
NEGACIONES = {
    "no", "nunca", "jamas", "tampoco",
    "nada", "nadie",
    "ningun", "ninguno", "ninguna", "ningunos", "ningunas",
    "ni", "sino", "sin",
}

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]


def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def extraer_features_linguisticos(texto):
    texto_str = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras = re.findall(r"\b\w+\b", texto_norm)

    n_exc = texto_str.count("!") + texto_str.count("¡")
    n_int = texto_str.count("?") + texto_str.count("¿")
    n_may = len(
        re.findall(r"\b[A-ZÁÉÍÓÚÜÑ]{3,}\b", texto_str)
    )
    n_emo = sum(
        1 for c in texto_str
        if c in emoji.EMOJI_DATA
    )
    n_ris = len(
        re.findall(
            r"\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b",
            texto_norm,
        )
    )
    n_neg = sum(
        1 for p in palabras
        if p in NEGACIONES
    )
    n_elo = len(
        re.findall(r"(.)\1{2,}", texto_norm)
    )
    n_com = len(
        re.findall(r'["\'‘’“”«»]', texto_str)
    )
    n_pun = texto_str.count("...") + texto_str.count("…")

    return [
        n_exc, n_int, n_may, n_emo, n_ris,
        n_neg, n_elo, n_com, n_pun,
    ]


for nombre, df in [("Train", df_train), ("Test", df_test)]:
    features = df["MESSAGE"].fillna("").apply(
        extraer_features_linguisticos
    ).tolist()

    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in features]

    print(f"{nombre}: features lingüísticas calculadas.")


Train: features lingüísticas calculadas.
Test: features lingüísticas calculadas.


## 6. Preprocessing BASE para BETO y DistilBETO

Esta versión realiza una limpieza mínima:

- Unicode y espacios normalizados.
- URL → `url`.
- mención → `@usuario`.
- elimina únicamente marcadores aislados `RT` / `FAV`.

Se conservan:

- mayúsculas;
- tildes;
- puntuación;
- `¿?` / `¡!`;
- hashtags;
- emojis;
- elongaciones;
- stopwords.

**DistilBETO es uncased:** no hace falta convertir manualmente el texto a minúsculas; se deja esa responsabilidad a su tokenizer oficial.


In [7]:
URL_RE = re.compile(
    r"(?:https?://\S+|www\.\S+)",
    flags=re.IGNORECASE,
)

MENTION_RE = re.compile(r"(?<!\w)@\w+")

RT_FAV_RE = re.compile(
    r"\b(?:RT|FAV)\b",
    flags=re.IGNORECASE,
)


def normalizar_unicode_y_espacios(texto):
    texto = unicodedata.normalize(
        "NFKC",
        str(texto),
    )

    texto = "".join(
        c for c in texto
        if unicodedata.category(c)[0] != "C"
        or c in "\n\t"
    )

    return re.sub(r"\s+", " ", texto).strip()


def preprocesar_transformer_base(texto):
    texto = normalizar_unicode_y_espacios(texto)

    texto = URL_RE.sub(
        " url ",
        texto,
    )

    texto = MENTION_RE.sub(
        " @usuario ",
        texto,
    )

    texto = RT_FAV_RE.sub(
        " ",
        texto,
    )

    return re.sub(r"\s+", " ", texto).strip()


## 7. Preprocessing SOCIAL para BETO y DistilBETO

La literatura sobre Twitter en español deja abiertas algunas decisiones para BETO/DistilBETO.

Por eso esta segunda variante permite comprobar posteriormente, mediante validación, si ayuda:

- limitar elongaciones a 3 caracteres;
- transformar hashtags en contenido textual;
- convertir emojis a descripción textual en español.

La puntuación y las stopwords se siguen conservando.


In [8]:
def limitar_elongaciones(
    texto,
    max_repeticiones=3,
):
    patron = re.compile(
        r"(.)\1{" + str(max_repeticiones) + r",}",
        flags=re.IGNORECASE,
    )

    return patron.sub(
        lambda m: m.group(1) * max_repeticiones,
        texto,
    )


def separar_camel_case(token):
    token = token.replace("_", " ")

    token = re.sub(
        r"(?<=[a-záéíóúüñ])(?=[A-ZÁÉÍÓÚÜÑ])",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])(?=\d)",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=\d)(?=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])",
        " ",
        token,
    )

    return re.sub(
        r"\s+",
        " ",
        token,
    ).strip()


def normalizar_hashtags(texto):
    def reemplazar(match):
        contenido = separar_camel_case(
            match.group(1)
        )

        return f" hashtag {contenido} "

    return re.sub(
        r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)",
        reemplazar,
        texto,
    )


def emojis_a_texto_es(texto):
    # Ejemplo aproximado:
    # 🙄 -> :cara_con_ojos_en_blanco:
    texto = emoji.demojize(
        texto,
        language="es",
    )

    patron = re.compile(
        r":([^:\s]+):"
    )

    def reemplazar(match):
        descripcion = (
            match.group(1)
            .replace("_", " ")
        )

        return (
            f" emoji {descripcion} emoji "
        )

    return patron.sub(
        reemplazar,
        texto,
    )


def preprocesar_transformer_social(texto):
    texto = preprocesar_transformer_base(
        texto
    )

    texto = limitar_elongaciones(
        texto,
        max_repeticiones=3,
    )

    texto = normalizar_hashtags(
        texto
    )

    texto = emojis_a_texto_es(
        texto
    )

    return re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()


## 8. RoBERTuito — preprocessing oficial

RoBERTuito fue entrenado específicamente para redes sociales en español.

Se utiliza directamente:

```python
pysentimiento.preprocessing.preprocess_tweet
```

en lugar de replicar manualmente su lógica.


In [9]:
def preprocesar_robertuito(texto):
    if not PYSENTIMIENTO_DISPONIBLE:
        raise ImportError(
            "Falta pysentimiento. Instala con "
            "`pip install pysentimiento` o "
            "`uv add pysentimiento`."
        )

    return preprocess_tweet(
        str(texto),
        lang="es",
        shorten=3,
    )


## 9. Generación de textos Transformer

In [10]:
for nombre, df in [
    ("Train", df_train),
    ("Test", df_test),
]:
    print(f"Procesando {nombre}...")

    df["MESSAGE_TRANSFORMER_BASE"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_base)
    )

    df["MESSAGE_TRANSFORMER_SOCIAL"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_social)
    )

    if PYSENTIMIENTO_DISPONIBLE:
        df["MESSAGE_ROBERTUITO"] = (
            df["MESSAGE"]
            .fillna("")
            .astype(str)
            .apply(preprocesar_robertuito)
        )
    else:
        df["MESSAGE_ROBERTUITO"] = pd.NA
        print(
            "  AVISO: RoBERTuito no se procesó "
            "porque falta pysentimiento."
        )

print("\nEjemplos por variante:")

for variante in ["mx", "es", "cu"]:
    row = (
        df_train[
            df_train["VARIANTE"] == variante
        ]
        .iloc[0]
    )

    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:")
    print(row["MESSAGE"])

    print("\nBASE:")
    print(row["MESSAGE_TRANSFORMER_BASE"])

    print("\nSOCIAL:")
    print(row["MESSAGE_TRANSFORMER_SOCIAL"])

    if PYSENTIMIENTO_DISPONIBLE:
        print("\nROBERTUITO:")
        print(row["MESSAGE_ROBERTUITO"])


Procesando Train...
  AVISO: RoBERTuito no se procesó porque falta pysentimiento.
Procesando Test...
  AVISO: RoBERTuito no se procesó porque falta pysentimiento.

Ejemplos por variante:

--- MX ---
ORIGINAL:
Rica económicamente, pero muy pobre en objetividad.

BASE:
Rica económicamente, pero muy pobre en objetividad.

SOCIAL:
Rica económicamente, pero muy pobre en objetividad.

--- ES ---
ORIGINAL:
@ArmandoRuido007 @ANTI_MERMA50 @JoanTarda En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣

BASE:
@usuario @usuario @usuario En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca 🤣🤣

SOCIAL:
@usuario @usuario @usuario En vez de Joan Tarda van a llamarle “No han tarda” en callarle la boca emoji cara revolviéndose de la risa emoji emoji cara revolviéndose de la risa emoji

--- CU ---
ORIGINAL:
magnifico

BASE:
magnifico

SOCIAL:
magnifico


## 10. Diagnóstico de tokenización — SOLO TRAIN

Este análisis debe ejecutarse **antes de decidir `max_length`**.

Para cada tokenizer, variante dialectal y preprocessing se calcula:

- P50;
- P95;
- P99;
- máximo;
- porcentaje > 64 tokens;
- porcentaje > 128 tokens;
- tokens por palabra;
- frecuencia de `[UNK]`, cuando aplica.

El test oficial no participa.


In [11]:
def cargar_tokenizers():
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "Falta transformers. Instala con "
            "`pip install transformers` o "
            "`uv add transformers`."
        )

    return {
        nombre: AutoTokenizer.from_pretrained(
            checkpoint
        )
        for nombre, checkpoint
        in CHECKPOINTS.items()
    }


def calcular_stats_tokenizacion(
    tokenizer,
    textos,
):
    longitudes = []
    unk = 0
    total_tokens = 0
    total_palabras = 0

    unk_id = tokenizer.unk_token_id

    for texto in textos:
        texto = str(texto)

        encoded = tokenizer(
            texto,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )

        ids = encoded["input_ids"]

        longitudes.append(
            len(ids)
        )

        total_tokens += len(ids)
        total_palabras += max(
            len(texto.split()),
            1,
        )

        if unk_id is not None:
            unk += sum(
                token == unk_id
                for token in ids
            )

    arr = np.asarray(
        longitudes,
        dtype=np.int32,
    )

    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "p99": np.percentile(arr, 99),
        "max": int(arr.max()),
        "pct_gt_64": 100 * (arr > 64).mean(),
        "pct_gt_128": 100 * (arr > 128).mean(),
        "tokens_por_palabra": (
            total_tokens
            / max(total_palabras, 1)
        ),
        "unk_por_1000_tokens": (
            1000 * unk
            / max(total_tokens, 1)
        ),
    }


def diagnosticar_train():
    tokenizers = cargar_tokenizers()

    configs = [
        (
            "BETO_BASE",
            "BETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "BETO_SOCIAL",
            "BETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
        (
            "DISTILBETO_BASE",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "DISTILBETO_SOCIAL",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
    ]

    if PYSENTIMIENTO_DISPONIBLE:
        configs.append(
            (
                "ROBERTUITO",
                "RoBERTuito",
                "MESSAGE_ROBERTUITO",
            )
        )

    filas = []

    for cfg, tokenizer_name, columna in configs:
        tok = tokenizers[tokenizer_name]

        for variante in ["mx", "es", "cu"]:
            textos = (
                df_train.loc[
                    df_train["VARIANTE"] == variante,
                    columna,
                ]
                .fillna("")
                .astype(str)
                .tolist()
            )

            stats = calcular_stats_tokenizacion(
                tok,
                textos,
            )

            filas.append({
                "configuracion": cfg,
                "tokenizer": tokenizer_name,
                "variante": variante,
                **stats,
            })

    return pd.DataFrame(filas)


if TRANSFORMERS_DISPONIBLE:
    df_tokenizacion = diagnosticar_train()

    display(
        df_tokenizacion.style.format({
            "p50": "{:.1f}",
            "p95": "{:.1f}",
            "p99": "{:.1f}",
            "pct_gt_64": "{:.2f}%",
            "pct_gt_128": "{:.2f}%",
            "tokens_por_palabra": "{:.3f}",
            "unk_por_1000_tokens": "{:.3f}",
        })
    )
else:
    df_tokenizacion = None
    print(
        "Instala transformers para ejecutar "
        "el diagnóstico."
    )


Instala transformers para ejecutar el diagnóstico.


## 11. Exportación por variante

Se genera un archivo por split y variante:

```text
data/transformers/
├── train_transformers_mx.csv
├── train_transformers_es.csv
├── train_transformers_cu.csv
├── test_transformers_mx.csv
├── test_transformers_es.csv
└── test_transformers_cu.csv
```

El diagnóstico de tokenización se guarda por separado y contiene **solo estadísticas del train**.


In [12]:
COLUMNAS_TEXTO = [
    "MESSAGE_TRANSFORMER_BASE",
    "MESSAGE_TRANSFORMER_SOCIAL",
    "MESSAGE_ROBERTUITO",
]

COLUMNAS_SALIDA = (
    [
        "ID",
        "TOPIC",
        "IS_IRONIC",
        "MESSAGE",
        "VARIANTE",
    ]
    + FEATURE_COLS
    + COLUMNAS_TEXTO
)

for split, df in [
    ("train", df_train),
    ("test", df_test),
]:
    for variante in ["mx", "es", "cu"]:
        salida = (
            df.loc[
                df["VARIANTE"] == variante,
                COLUMNAS_SALIDA,
            ]
            .reset_index(drop=True)
        )

        ruta = (
            OUT_DIR
            / f"{split}_transformers_{variante}.csv"
        )

        salida.to_csv(
            ruta,
            index=False,
        )

        print(
            f"{ruta} -> "
            f"{len(salida)} registros"
        )

if df_tokenizacion is not None:
    ruta_diag = (
        OUT_DIR
        / "diagnostico_tokenizacion_train.csv"
    )

    df_tokenizacion.to_csv(
        ruta_diag,
        index=False,
    )

    print(
        "\nDiagnóstico:",
        ruta_diag,
    )


# Guardar auditorías de calidad del dataset.
df_auditoria_train.to_csv(
    OUT_DIR / "auditoria_duplicados_train.csv",
    index=False,
)

df_auditoria_train_test.to_csv(
    OUT_DIR / "auditoria_train_test.csv",
    index=False,
)

print(
    "\nAuditorías guardadas en:",
    OUT_DIR,
)


# Registros retirados del train por coincidencia exacta con test.
if len(df_train_excluidos) > 0:
    df_train_excluidos.to_csv(
        OUT_DIR / "train_excluidos_por_solapamiento_test.csv",
        index=False,
    )

# Detalle completo train-test por variante.
for variante, detalle in coincidencias_train_test.items():
    if not detalle.empty:
        detalle.to_csv(
            OUT_DIR / f"coincidencias_train_test_{variante}.csv",
            index=False,
        )


# Guardar filas excluidas por conflicto de anotación.
if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(
        OUT_DIR / "conflictos_etiqueta_train.csv",
        index=False,
    )


..\data\transformers\train_transformers_mx.csv -> 2397 registros
..\data\transformers\train_transformers_es.csv -> 2396 registros
..\data\transformers\train_transformers_cu.csv -> 2400 registros
..\data\transformers\test_transformers_mx.csv -> 600 registros
..\data\transformers\test_transformers_es.csv -> 600 registros
..\data\transformers\test_transformers_cu.csv -> 600 registros

Auditorías guardadas en: ..\data\transformers


## 12. Uso posterior

En los notebooks de entrenamiento:

### BETO
Comparar dentro de validación:

```text
MESSAGE_TRANSFORMER_BASE
vs.
MESSAGE_TRANSFORMER_SOCIAL
```

### DistilBETO
Comparar igualmente:

```text
MESSAGE_TRANSFORMER_BASE
vs.
MESSAGE_TRANSFORMER_SOCIAL
```

El tokenizer oficial se encarga del lowercasing.

### RoBERTuito
Usar:

```text
MESSAGE_ROBERTUITO
```

El checkpoint `deacc` puede compararse posteriormente contra `uncased` usando el mismo protocolo de validación.

### Importante

El test oficial no se utiliza para decidir:

- BASE vs SOCIAL;
- RoBERTuito uncased vs deacc;
- max_length;
- hiperparámetros;
- mejor checkpoint.


### Validación y entrenamiento

Este notebook **no ejecuta Cross-Validation ni Nested Cross-Validation**.

Eso se implementa en los notebooks de entrenamiento:

```text
BETO / DistilBETO / RoBERTuito
        ↓
Nested CV 5×3
        ↓
Outer CV = 5 folds
Inner CV = 3 folds
        ↓
selección por F1-Macro
```

El preprocessing únicamente prepara y audita los datos.


## 13. Preprocessing definitivo utilizado

### Control de calidad
1. Auditar duplicados internos del train.
2. Si un mismo texto tiene etiquetas contradictorias, retirar todas sus apariciones del train y registrarlas en auditoría.
3. Eliminar duplicados exactos del train dentro de cada variante.
4. Auditar coincidencias exactas train↔test.
5. Mantener el test oficial intacto y retirar del train los textos que aparecen exactamente en test.

### BETO cased
Se compararán posteriormente, dentro del **Inner CV**, dos entradas:

- `MESSAGE_TRANSFORMER_BASE`: Unicode/espacios + URL y usuario normalizados; conserva casing, acentos, puntuación, emojis, hashtags, stopwords y elongaciones.
- `MESSAGE_TRANSFORMER_SOCIAL`: BASE + elongaciones ≤3 + hashtag a contenido textual + emoji a descripción española.

No se aplica stemming ni lematización.

### DistilBETO uncased
Utiliza las mismas variantes BASE/SOCIAL.  
No se fuerza lowercase manualmente: se utiliza el tokenizer oficial del checkpoint uncased.

### RoBERTuito
Se utiliza exclusivamente `MESSAGE_ROBERTUITO`, generado con
`pysentimiento.preprocessing.preprocess_tweet`.

### Longitud
`max_length = 128` será el punto de partida para entrenamiento.
El diagnóstico de tokenización se calcula exclusivamente sobre train.

### Validación
La selección BASE vs SOCIAL, hiperparámetros y checkpoints no se realiza en este notebook.
Se hará posteriormente mediante **Nested CV 5×3** en los notebooks de entrenamiento.
